In [14]:
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [2]:
X = make_moons(n_samples=10000, noise=0.4)
X_train, X_test, y_train, y_test = train_test_split(X[0], X[1], test_size=0.2, random_state=42)

In [ ]:
log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')

voting_clf.fit(X_train, y_train)

In [4]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.836
RandomForestClassifier 0.851
SVC 0.8675
VotingClassifier 0.8635


In [5]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=100, bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

In [6]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    bootstrap=True, n_jobs=-1, oob_score=True)

bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.839875

In [7]:
bag_clf.oob_decision_function_

array([[0.01030928, 0.98969072],
       [1.        , 0.        ],
       [0.01666667, 0.98333333],
       ...,
       [0.92473118, 0.07526882],
       [0.98994975, 0.01005025],
       [0.07734807, 0.92265193]])

In [8]:
rnd_clf = RandomForestClassifier(
    n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)

In [9]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(splitter="random", max_leaf_nodes=16),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)

In [10]:
from sklearn.datasets import load_iris
iris = load_iris()
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris["data"], iris["target"])

for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

sepal length (cm) 0.09961864634670145
sepal width (cm) 0.02502061403128628
petal length (cm) 0.4293887348224385
petal width (cm) 0.44597200479957383


In [ ]:
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    algorithm="SAMME", learning_rate=0.5)
ada_clf.fit(X_train, y_train)

In [20]:
X = iris['data'][:, 2:]
y = iris['target']

In [ ]:
tree_reg1 = DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(X, y)

y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(X, y2)

y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(X, y3)

In [27]:
y_pred = sum(tree.predict(X) for tree in (tree_reg1, tree_reg2, tree_reg3))

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2,
                                 n_estimators=3,
                                 learning_rate=1.0)
gbrt.fit(X, y)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np


X_train, X_val, y_train, y_val = train_test_split(X, y)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120)
gbrt.fit(X_train, y_train)

errors = ([mean_squared_error(y_val, y_pred)
          for y_pred in gbrt.staged_predict(X_val)])
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators)
gbrt_best.fit(X_train, y_train)

In [30]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True)

min_val_error = float("inf")
error_going_up = 0
for n_estimators in range(1, 120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_val)
    val_error = mean_squared_error(y_val, y_pred)

    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            break

In [33]:
import xgboost

xgb_reg = xgboost.XGBRegressor()
xgb_reg.fit(X_train, y_train,
            eval_set=[(X_val, y_val)])
y_pred = xgb_reg.predict(X_val)

mean_squared_error(y_val, y_pred)

[0]	validation_0-rmse:0.57004
[1]	validation_0-rmse:0.41765
[2]	validation_0-rmse:0.31223
[3]	validation_0-rmse:0.24892
[4]	validation_0-rmse:0.21311
[5]	validation_0-rmse:0.19106
[6]	validation_0-rmse:0.18249
[7]	validation_0-rmse:0.17936
[8]	validation_0-rmse:0.17912
[9]	validation_0-rmse:0.17978
[10]	validation_0-rmse:0.18085
[11]	validation_0-rmse:0.18201
[12]	validation_0-rmse:0.18310
[13]	validation_0-rmse:0.18407
[14]	validation_0-rmse:0.18492
[15]	validation_0-rmse:0.18564
[16]	validation_0-rmse:0.18626
[17]	validation_0-rmse:0.18678
[18]	validation_0-rmse:0.18725
[19]	validation_0-rmse:0.18764
[20]	validation_0-rmse:0.18798
[21]	validation_0-rmse:0.18826
[22]	validation_0-rmse:0.18849
[23]	validation_0-rmse:0.18869
[24]	validation_0-rmse:0.18886
[25]	validation_0-rmse:0.18902
[26]	validation_0-rmse:0.18914
[27]	validation_0-rmse:0.18926
[28]	validation_0-rmse:0.18935
[29]	validation_0-rmse:0.18942
[30]	validation_0-rmse:0.18949
[31]	validation_0-rmse:0.18952
[32]	validation_0-

0.03592536225914955